# 5장 실습 — GTFS 데이터 조회와 시간표 구축

GTFS는 운영기관·정류장·노선·시간표·운행일을 공통 형식으로 전달하는 공개 표준입니다.
Google 지도 같은 서비스가 이 데이터를 받아 대중교통 정보를 안내합니다.
먼저 교재 5.1절에서 파일별 역할과 Schedule·Realtime의 차이를 읽습니다.
이 실습은 예정 시간표인 Schedule을 사용합니다.

교재 5.2~5.5절의 시간표를 읽고 새 출발편을 작성합니다.
먼저 시간표를 손으로 읽고, 코드 결과를 예상하고, 실행한 표를 설명합니다.
마지막에는 A→B→C를 지나는 자기 셔틀의 출발편 두 개를 만듭니다.

- 읽을 예제: `L1`, A 08:00·08:30 출발, 구간 10분·10분.
- 만들 예제: `MY`, A 08:00·08:20 출발, 구간 4분·6분.

하남 전체 자료·지도는 [추가 탐색](extensions/ch05_gtfs_exploration.ipynb)에,
ZIP 저장과 노선 추가 비교는 [통합 과제](../projects/gtfs_route_design/route_design.ipynb)에 있습니다.
이 노트북에서는 작은 표의 한 행이 뜻하는 것부터 확인합니다. 아래 셀은 실행 환경을 준비합니다.

실습에서 읽는 `stops`, `routes`, `trips`, `stop_times`, `calendar`는 표 이름을 키로 쓰는 자료입니다.
운영기관 `agency`는 독립 ZIP을 만드는 통합 과제에서 추가합니다. 운행 날짜 예외는 이번 작은 예제에 없습니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

import pandas as pd
from lab import expect, todo
from smartmob.data import parse_gtfs_time, seconds_to_gtfs_time
from smartmob.teaching.raptor import toy_feed

## 1. 노선·운행·정류장 방문의 구분 (교재 5.2)

먼저 종이에 A→B→C를 쓰고, 아래에 08:00→08:10→08:20과 08:30→08:40→08:50 두 줄을 적습니다.
노선은 1개, 운행은 2편, 방문은 6번입니다. 정류장은 여전히 3곳입니다.
`toy_feed()`에는 다음 장의 2호선도 들어 있으므로 지금은 1호선만 골라 읽습니다.

In [ ]:
source = toy_feed()
read_trips = source["trips"].query("route_id == 'L1'").copy()
read_times = source["stop_times"][source["stop_times"]["trip_id"].isin(read_trips["trip_id"])].copy()
read_stops = source["stops"][source["stops"]["stop_id"].isin(["A", "B", "C"])].copy()
read_routes = source["routes"].query("route_id == 'L1'").copy()
print("노선", len(read_routes), "운행", len(read_trips), "방문", len(read_times))
read_times.pivot(index="trip_id", columns="stop_id", values="departure_time")

출력은 노선 1개·운행 2편·방문 6행입니다. 가로로 읽으면 한 차의 시간표이고 세로로 읽으면 한 정류장의 출발편입니다.
A 08:03 승객이 읽을 행을 골라 봅시다. 이미 떠난 `L1-1`을 제외하면 `L1-2`가 남습니다.

## 2. 식별자를 이용한 시간표와 정류장 정보 결합 (교재 5.2)

`trip_id`로 출발편 하나를 고르면 방문 세 행이 남습니다.
`stop_id`를 정류장 표에서 찾아 이름을 붙이고 `stop_sequence`로 정렬합니다.
실행 전에 `L1-2`의 B 도착시각 08:40을 표에서 먼저 찾아 봅니다.

In [ ]:
selected_trip = "L1-2"
visits = read_times[read_times["trip_id"] == selected_trip]
timetable = visits.merge(read_stops, on="stop_id", validate="many_to_one")
timetable = timetable.sort_values("stop_sequence")
timetable[["trip_id", "stop_sequence", "stop_name", "arrival_time", "departure_time"]]

A역 08:30, B역 08:40, C역 08:50의 세 행입니다. 이름을 붙여도 방문 수는 늘지 않습니다.
이번에는 방문 시각에서 출발편 표를 거쳐 노선 이름까지 연결합니다.
각 방문의 `trip_id`가 어느 `route_id`를 가리키는지 손으로 따라간 뒤 실행합니다.

In [ ]:
named = read_times.merge(read_trips, on="trip_id", validate="many_to_one")
named = named.merge(read_routes, on="route_id", validate="many_to_one")
assert len(named) == 6
named[["route_short_name", "trip_id", "stop_id", "stop_sequence", "departure_time"]]

두 운행 모두 1호선이며 각 운행 안에서 방문 순서는 다시 1부터 시작합니다.
`trip_id=L1-2, stop_id=B, stop_sequence=2`인 행을 한 문장으로 설명해 봅니다.

## 3. 운행일 판별과 시간 계산 (교재 5.3)

시각표를 쓰기 전에 두 편이 언제 운행하는지 정합니다. 이번에는 2026-09-23 하루로 제한합니다.
`S`는 두 운행이 함께 참조하는 규칙입니다. 요일이 모두 1이어도 시작일·종료일 밖에서는 운행하지 않습니다.

In [ ]:
days = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
read_calendar = pd.DataFrame([{
    "service_id": "S", **{day: 1 for day in days},
    "start_date": "20260923", "end_date": "20260923",
}])
assert set(read_trips["service_id"]) <= set(read_calendar["service_id"])
read_calendar

`S` 한 행이 두 운행에 연결됩니다. 다음 날도 탈 수 있는지 물으면 적용 종료일을 확인해야 합니다.
날짜 조건을 확인했으므로 이제 08:03 승객의 대기·차내·합계 시간을 계산합니다.
계산 전 예상은 27분·20분·47분입니다.

In [ ]:
ready_s = parse_gtfs_time("08:03:00")
board_s = parse_gtfs_time("08:30:00")
arrive_s = parse_gtfs_time("08:50:00")
print("대기:", (board_s - ready_s) / 60, "분")
print("차내:", (arrive_s - board_s) / 60, "분")
print("합계:", (arrive_s - ready_s) / 60, "분")

08:50은 도착시각이고 47분은 걸린 시간입니다. 둘을 같은 값처럼 비교하지 않습니다.
자정을 넘는 23:50→다음 날 00:20도 30분이 되는지 확인합니다.
GTFS의 도착 표기는 `24:20:00`입니다.

In [ ]:
night_departure = parse_gtfs_time("23:50:00")
night_arrival = parse_gtfs_time("24:20:00")
print("출발·도착 초:", night_departure, night_arrival)
print("통행시간:", (night_arrival - night_departure) / 60, "분")
print("다시 시각으로:", seconds_to_gtfs_time(night_arrival))

85,800초에서 87,600초까지 1,800초, 즉 30분입니다. 변환 후에도 `24:20:00`이 유지됩니다.
세 표의 역할과 대기시간을 설명할 수 있으면 새 노선을 만듭니다.

## 4. 신규 노선의 운행 시간표 작성 (교재 5.4)

A→B는 4분, B→C는 6분입니다. A에서 출발한 뒤 B까지 4분, C까지는 누적 10분입니다.
첫 편은 08:00, 두 번째 편은 08:20 출발하며 정차시간은 0초입니다.
`stops`와 규칙 `S`를 재사용하고 노선·운행 식별자를 새로 만듭니다.
아래 표를 실행하기 전에 `routes`는 1행, `trips`는 2행이 될 이유를 설명합니다.

In [ ]:
my_routes = pd.DataFrame([("MY", "내 셔틀", 3)],
                         columns=["route_id", "route_short_name", "route_type"])
my_trips = pd.DataFrame([("MY", "S", "MY-1"), ("MY", "S", "MY-2")],
                        columns=["route_id", "service_id", "trip_id"])
display(my_routes, my_trips)

`MY-1`과 `MY-2`는 같은 노선 `MY`와 운행일 `S`를 씁니다. 신규 버스의 표준 수단 코드는 3입니다.
이제 첫 편의 방문 세 행을 읽습니다. 각 튜플은 `(운행 식별자, 정류장, 방문 순서, 시각)` 순서입니다.

In [ ]:
first_visits = [
    ("MY-1", "A", 1, "08:00:00"),
    ("MY-1", "B", 2, "08:04:00"),
    ("MY-1", "C", 3, "08:10:00"),
]
visit_columns = ["trip_id", "stop_id", "stop_sequence", "arrival_time"]
pd.DataFrame(first_visits, columns=visit_columns)

B는 08:04, C는 08:10입니다. C의 시각에는 4분과 6분을 모두 더했습니다.
두 번째 편은 08:20→08:24→08:30입니다. 먼저 종이에 쓴 뒤 아래 `second_visits`에 같은 형태의 튜플 세 개를 넣습니다.
운행 식별자는 `MY-2`이고 방문 순서는 다시 1·2·3입니다. `None`이면 뒤의 작성·조회는 안내만 출력합니다.

In [ ]:
second_visits = None
my_stop_times = None

todo("두 번째 출발편의 방문 세 행", second_visits)
if second_visits is not None:
    my_stop_times = pd.DataFrame(first_visits + second_visits, columns=visit_columns)
    my_stop_times["departure_time"] = my_stop_times["arrival_time"]
    display(my_stop_times)
else:
    print("종이에 쓴 MY-2의 세 행을 second_visits에 넣습니다.")

작성 후 여섯 방문 행이 나옵니다. 정차시간이 0초여서 도착과 출발 컬럼을 같게 두었습니다.
개수만 맞아도 식별자나 시각이 잘못될 수 있습니다. 각 운행이 A→B→C를 방문하고 구간시간을 지키는지 검사합니다.

In [ ]:
if my_stop_times is not None:
    assert len(my_stop_times) == 6
    assert set(my_stop_times["trip_id"]) == set(my_trips["trip_id"])
    assert set(my_stop_times["stop_id"]) <= set(read_stops["stop_id"])
    for trip_id, group in my_stop_times.groupby("trip_id"):
        group = group.sort_values("stop_sequence")
        assert group["stop_sequence"].tolist() == [1, 2, 3]
        assert group["stop_id"].tolist() == ["A", "B", "C"]
        times = group["arrival_time"].map(parse_gtfs_time).tolist()
        assert [times[1] - times[0], times[2] - times[1]] == [4 * 60, 6 * 60]
    assert my_stop_times.query("trip_id == 'MY-2'").sort_values("stop_sequence").iloc[0]["arrival_time"] == "08:20:00"
    print("운행·정류장 연결과 두 구간의 시간을 확인했습니다.")

두 운행 모두 구간 4분·6분으로 이어지는지 확인했습니다.
이 검사는 작은 시간표의 조건을 확인하며, GTFS 전체 명세 검사와는 구분합니다.

## 5. 탑승 가능 운행과 도착시각 검증 (교재 5.5)

A 08:00 승객은 `MY-1`로 C 08:10, A 08:03 승객은 `MY-2`로 C 08:30에 도착합니다.
A 08:21 승객은 이 시간표 안에서 탈 차가 없습니다.
세 경우를 설명한 뒤 코드로 확인합니다. 먼저 같은 운행의 A 출발과 C 도착을 한 행으로 붙입니다.

In [ ]:
if my_stop_times is not None:
    at_a = my_stop_times.query("stop_id == 'A'")[["trip_id", "departure_time"]]
    at_c = my_stop_times.query("stop_id == 'C'")[["trip_id", "arrival_time"]]
    direct = at_a.merge(at_c, on="trip_id", validate="one_to_one")
    direct["dep_s"] = direct["departure_time"].map(parse_gtfs_time)
    direct["arr_s"] = direct["arrival_time"].map(parse_gtfs_time)
    display(direct)
else:
    print("4절의 두 번째 출발편을 작성한 뒤 조회합니다.")

방문 여섯 행이 운행별 두 행으로 정리됩니다. 각 행은 A에서 탄 차를 C까지 따라간 경우입니다.
준비 시각 이후의 출발만 남기고 가장 먼저 떠나는 차를 선택합니다.

In [ ]:
if my_stop_times is not None:
    answers = []
    for ready in ["08:00:00", "08:03:00", "08:21:00"]:
        ready_s = parse_gtfs_time(ready)
        options = direct[direct["dep_s"] >= ready_s].sort_values("dep_s")
        row = {"준비 시각": ready, "운행": "없음", "도착": "미도달",
               "대기(분)": None, "차내(분)": None, "합계(분)": None}
        if not options.empty:
            choice = options.iloc[0]
            row.update({"운행": choice.trip_id, "도착": choice.arrival_time,
                        "대기(분)": (choice.dep_s - ready_s) / 60,
                        "차내(분)": (choice.arr_s - choice.dep_s) / 60,
                        "합계(분)": (choice.arr_s - ready_s) / 60})
        answers.append(row)
    pd_result = pd.DataFrame(answers)
    display(pd_result)

08:03의 대기 17분과 차내 10분을 더하면 27분입니다. 미도달 행은 시간을 빈값으로 남깁니다.
제출물은 시간표 여섯 행과 설계 이유 두 문장입니다.

마무리 질문 ★: 다음 편을 08:10으로 옮기면 B·C 시각은 무엇일까요?
답은 08:14·08:20입니다. 출발만 고치지 않고 그 운행의 세 방문 시각을 함께 바꿔야 합니다.
기본안 확인용 검사에는 08:20이 적혀 있으므로, 변경 실험은 기본안을 제출한 뒤 별도 셀에서 진행합니다.

6장에서는 1호선 예제에 도보 환승과 2호선을 더합니다.
하남 전체 자료는 [추가 탐색](extensions/ch05_gtfs_exploration.ipynb),
노선 ZIP과 효과 비교는 6장을 마친 뒤 [통합 과제](../projects/gtfs_route_design/route_design.ipynb)에서 진행합니다.